# T1-05 Use-case canvas (group)

**Time:** 14:20–14:55, then overnight 20–30 min &nbsp;|&nbsp; **Evidence:** canvas (`cards/usecase_canvas.md`) + gold pairs (logged)

**Notebook learning outcomes**

Write a testable use case a domain person can verify. Frame the *type* of problem in the EOI, not a live system. Watch one worked canvas get measured, then design your own experiment: what you would measure, what counts as good enough, and what stops a document.

**This block is (group).** Ministry groups of three to four. A singleton joins the same learning-need cluster. Each member still has this notebook open.


**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the teaching note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** run all cells. Change only lines marked `# [CHANGE]`.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path. Do not drive someone else's notebook.

Git is not required. If you are stuck, load the matching file in `completed/` for this notebook only, mark it as loaded from reference, and continue.


## This block is design, not build

This is a thought exercise in **experimental design**. Nobody builds a system here.

- You do **not** need your own dataset today. Do not fetch one.
- Your job is to design the experiment and pin down the metrics: who checks the output, what "wrong" looks like, how you would count it, what number is good enough (written down **before** any run), and what makes a single document stop.
- The instructor's worked example runs a real measurement so you can see what your design turns into later. You copy the *shape*, not the task.


## Ministry link — pick the row for your table

| Cluster | Sit with | Canvas must include |
|---|---|---|
| **A Criteria** | Labour, Health | Published criteria; quote or flag; **human review**; stop = no hire / no published grade |
| **B Retrieval** | Finance, Defence OPR, MPSDFS policy, MOWI ECMS, RDLG minutes, THA | Approved corpus; cite or abstain; access rule (N4) |
| **C Service desk** | Trade IMSD, Defence assets-as-query | Triage; tools the model must **never** call; register vs generation (N10) |
| **D Local control** | CPO, TTDF, others with custody constraints | Where the box lives; what data never leaves; agentic “no” |

If the real job is GIS routes or radar patterns, say **maybe no language model** (AUX-02) on the canvas. That is a valid Day 1 outcome.

## Overnight readers

The morning was the machine. This is the **job**. Write a use case a domain person can mark wrong. You will not choose prompt vs RAG vs fine-tune until Day 2 (T1-06). Tonight: owner, verifier, baseline, measure, error cost, stop, data that stays out. CivicDesk names only — no live EOI attachments.


## Worked example: what Day 1 already gave us, and what it did not

The instructor's example is a fictional CivicDesk HR task: **remove personal data from CVs before an internal shortlisting panel reads them**, so the panel judges qualifications, not identity. Every CV is synthetic.

What the morning gave this project:

- **T1-01 first call:** we can send text to OLMo through the course pipeline and get text back. That is plumbing, not quality.
- **T1-02 tokens:** we can count what a CV costs in tokens, so latency and cost questions have a unit.
- **T1-03 model comparison:** we learned how to measure on a frozen, rule-labelled set. Its GO/BLOCK labels scored draft-reply screening, a **different task**. It says nothing about redaction.
- **T1-04 licence and GPU gates:** OLMo passed "may we" and "can we" for classroom use. That makes it the *provisional* model to measure, not the chosen one.

What we do not know yet: whether a prompted OLMo redacts CVs well enough. Only a measurement on this task can say.


In [ ]:
# [RUN] the instructor's worked canvas. Fictional CivicDesk names only.
worked = {
    "organisation": "CivicDesk HR Recruitment Unit (fictional)",
    "cluster": "A Criteria, with D custody constraints",
    "problem_in_one_line": "Before an internal shortlisting panel reads CVs for a vacancy, remove applicant and third-party "
                           "personal identifiers without changing any work content.",
    "owner_role": "Recruitment manager (owns the process and the go/no-go for each stage)",
    "domain_verifier_role": "Recruitment officer already authorised to see unredacted CVs; the data protection "
                            "officer rules on policy edge cases (e.g. a family business name)",
    "current_non_ai_baseline": "Officer redacts by hand in a word processor; a second officer spot-checks. "
                               "Minutes per CV and misses per 100 CVs: not yet measured. Measure on 20 synthetic CVs "
                               "before comparing.",
    "success_measure": "On the frozen 40-CV held-out set: missed PII by type (recall), documents with any leak, "
                       "over-redaction or altered text, empty/truncated outputs, latency, tokens, "
                       "cost per acceptable answer (scoped).",
    "acceptance_rule": "EXAMPLE, fixed before the run: 0 documents with a residual email/phone/ID/DOB; at most "
                       "2 of 35 with any leak; at most 4 of 40 over-redacted or altered; 0 empty or truncated; "
                       "p95 latency at most 20 s on the course GPU.",
    "test_set_plan": "40 synthetic CVs, 8 layouts fixed in advance, labels from code, frozen by hash. Never used "
                     "to train, tune prompts or pick examples. Day 3 reuses it unchanged.",
    "error_cost_if_wrong": "Missed PII: the panel sees identity (bias, privacy breach). Over-redaction or altered "
                           "text: the panel misjudges a candidate. Truncation: experience silently disappears.",
    "data_that_must_stay_out": "Real CVs, photos, scans, anything from the live HR system; no raw CV text in "
                               "shared logs.",
    "preferred_intervention_guess": "unsure. Measure prompted OLMo first (provisional). T1-06 argues prompt vs RAG "
                                    "vs fine-tune; Day 3 measures one adapter on this same set. No outcome assumed.",
    "stop_condition": "Per document: any residual email/phone/ID, a known name from the application form, empty, "
                      "much shorter than the input, timeout, out-of-scope input -> HOLD for authorised review, "
                      "not released. Whole service: pause on a failed audit sample, a burst of holds, suspected "
                      "drift or a new layout/language -> fall back to the manual process.",
    "one_cost_question": "Reviewer minutes per held CV, times the hold rate, versus today's manual minutes per CV.",
    "next_question_for_MPAAI": "Who owns the redaction policy, and may a family business name stay?",
}
for k, v in worked.items():
    print(f"{k}: {v}\n")


### Teaching note: a quality target is not a stop trigger

Two different things sit on the canvas.

- **Acceptance rule (quality target):** a pass mark for a *batch*, checked once on the frozen test set. It needs the answer key. It answers "is this good enough to continue to the next stage?"
- **Stop condition (per-document trigger):** checked on *every* document at run time, with no answer key. It answers "may this one output go to the panel, or does it wait for a person?" If the email check fires, that CV is held and routed to the authorised verifier. It is not released.

The checks have false alarms (a reference number that looks like a phone number) and misses (an email spelled out in words). **No alert does not mean safe.**


In [ ]:
# [RUN] seven worked CVs, scored against the answer key, then through the stop rule.
# The outputs below were WRITTEN BY HAND to show one failure each. No model produced them.
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..") / "shared" / "resume_redaction"))
import redaction_eval as R

teach = R.load_fixture()
replay = R.load_replay()
assert replay["fixture_sha256"] == teach["sha256"], "replay does not belong to this teaching set"
print(replay["label"], "\n")
rows = R.evaluate(teach, replay["outputs"])
for doc, r in zip(teach["docs"], rows):
    print(f"{r['id']} ({doc['layout']})")
    print(f"   shown on purpose: {replay['what_each_shows'][r['id']]}")
    print(f"   answer key says: missed={[m['type'] + ': ' + m['text'] for m in r['spans_missed']]}"
          f" errors={[e['kind'] for e in r['edits']]}")
    print(f"   run-time stop rule: {r['reasons'] or 'nothing fired'} -> {r['decision']}\n")


### Reading the replay

- **R05 is the lesson.** The email leaked and no check fired. A quiet run is not a safe run.
- **R07 is the cost.** A correct output was held. Every false hold is a person's time.
- **R03's 30% → 40%** is invisible to every run-time check. Only the answer key catches it, which is why you measure on a labelled set *before* trusting run-time checks.
- **R04** passed the checks and still harmed the candidate (the employer vanished). Over-redaction is an error with a cost, not a safe default.
- **R06** cut off mid-sentence. Span recall would count the cut-off name as "removed", so read recall beside truncation.

Adding a second detector or a second model does not automatically fix this. Two systems trained on similar text tend to miss the same spelled-out email.


In [ ]:
# [RUN] the frozen 40-CV benchmark for prompted OLMo.
# Learners: leave MODE = "replay" (loads the run measured in this container image).
# Instructor: MODE = "live" sends 40 requests to the backend chosen in T1-01. Same set, same prompt, same rule.
MODE = "replay"  # [CHANGE] instructor only: "live"

bench = R.load_benchmark()
if MODE == "live":
    import civicdesk_lab as lab
    run = R.run_live(bench, "olmo")
    if not run["measured"]:
        print("Not measured (demo or fallback replies). Showing the committed baseline instead.")
        run = R.load_run("olmo_baseline_v1")
else:
    run = R.load_run("olmo_baseline_v1")
assert run["fixture_sha256"] == bench["sha256"], "baseline was not run on this benchmark"
assert run["measured"], "baseline file is not a measured run"
print(f"model {run['model']} | prompt {run['prompt_version']} | benchmark {run['benchmark_version']} | run {run['run_at_utc']}")

rows = R.evaluate(bench, run["outputs"], run["latency_s"])
summary = R.summary_table(bench, rows)
for k, v in summary.items():
    print(f"{k}: {v}")

by_cat = {}
for d, r in zip(bench["docs"], rows):
    c = by_cat.setdefault(d["category"], [0, 0])
    c[0] += 1
    c[1] += r["acceptable"]
print("\nacceptable as-is by category:", {k: f"{v[1]}/{v[0]}" for k, v in by_cat.items()})

verdict = R.check_rule(rows, run["latency_s"])
print("\n" + verdict["rule"])
for name, value, ok in verdict["criteria"]:
    print(f"  {'ok  ' if ok else 'FAIL'} {name}: {value}")
print("Meets the example rule:", verdict["meets_rule"])


### What the baseline says, and what it does not

- It is a **measurement of one prompt on one model on 40 synthetic CVs**. It is the number Day 3 has to beat, and the same 40 CVs, scorer and rule come back on Day 3 unchanged.
- If it **meets** the rule, the honest conclusion is that a prompt may be enough for this classroom task. Fine-tuning would need another reason.
- If it **does not**, Day 3 tests whether one adapter closes the gap. It may help, it may not, and it may fix one category and break another. Nobody knows until it is measured.
- Either way, **meeting the rule is not permission to release anything.** It is synthetic, classroom evidence. Real CVs need a real test set, an authorised environment and the owner's decision.
- 40 documents is small. Zero failures in 40 is consistent with a true failure rate up to about 7% (one-sided 95% bound, and only for documents like these). Differences of one or two documents between runs are noise-sized.
- The rule was written **before** the run. Changing it after seeing the result is a new experiment, not a pass.


In [ ]:
# [CHANGE] fill as a group. Fictional CivicDesk names only. Design only: no dataset needed today.
canvas = {
    "organisation": "",
    "cluster": "A / B / C / D",
    "problem_in_one_line": "",
    "owner_role": "",
    "domain_verifier_role": "",
    "current_non_ai_baseline": "",
    "success_measure": "",
    "acceptance_rule": "write the pass mark BEFORE any run, as numbers",
    "test_set_plan": "how many examples, which categories, who labels them, how it stays frozen",
    "error_cost_if_wrong": "",
    "data_that_must_stay_out": "",
    "preferred_intervention_guess": "prompt / RAG / integration / fine-tune / none / unsure",
    "stop_condition": "",
    "one_cost_question": "",
    "next_question_for_MPAAI": "",
}
for k, v in canvas.items():
    print(f"{k}: {v}")


### Teaching note

A fluent paragraph is not a use case. A domain verifier must be able to say “that output is wrong” using a measure you wrote down. Labour: evidence quote vs criterion. Trade: ticket fields complete, password not reset. Finance: citation present or abstain.

Compare your canvas with the worked one, line for line. Yours does not need data, only a design: each measure should be something you could count, with a pass mark you would accept or reject before seeing a result, and a stop condition that works on one document at run time without an answer key.

You will **not** finish the intervention label today. Day 2 morning (T1-06) is that argument. Overnight you add: owner, verifier, baseline, measure, error cost, stop, and three remaining questions.

Copy the dict into `cards/usecase_canvas.md` before you leave. Download this notebook at 14:55.


In [ ]:
# [CHANGE] Gold pairs: for YOUR use case, write text that SHOULD trigger the action beside a near miss that
# should NOT. Invent the text. Fictional CivicDesk only. One pair in class, six by tomorrow.
# These are design drafts, not training data and not test data.
gold_pairs = [
    {
        "rule": "which line of your policy or criteria this tests",
        "should_trigger": "",
        "near_miss_should_not": "",
        "why_they_differ": "",
        "what_real_data_you_would_need": "",
    },
]

worked_pair = {  # the instructor's example, from the résumé task
    "rule": "PERSON: redact private individuals; keep employer names",
    "should_trigger": "Referee: Gail Ramsaran, Former Supervisor",
    "near_miss_should_not": "Accounts Clerk, Ramdass & Sons Hardware Ltd",
    "why_they_differ": "Same kind of word (a family name); one is a person, one is an employer the policy keeps",
    "what_real_data_you_would_need": "CVs where family names appear in employer names, labelled by the policy owner",
}

for i, p in enumerate(gold_pairs, 1):
    empty = [k for k, v in p.items() if not str(v).strip()]
    same = p["should_trigger"].strip() and p["should_trigger"].strip() == p["near_miss_should_not"].strip()
    print(f"pair {i}: " + ("complete" if not empty and not same else f"to finish: {empty or 'the two texts are identical'}"))


### Teaching note: pairs are your data roadmap

A model learns a boundary from examples on both sides of it. A near miss is where a system usually fails. Your pairs are the start of a data-preparation plan:

1. **Freeze the test set first.** Before anyone writes training examples, set aside held-out examples nobody trains on or tunes against.
2. **Each pair names a category** the test set must cover, and a category the training data must cover.
3. **Labels come from your policy, checked by your verifier,** not from a model marking its own work.
4. **Don't copy test items into training.** If a pair came from a failure you saw in the test set, write a new example of that *kind*. Never reuse the test text.
5. The column "what real data you would need" is your shopping list for the real project, and the list of what must stay out.


In [ ]:
# [RUN] save whatever you have. Empty strings are allowed at 14:55; overnight fills them.
import civicdesk_lab as lab
lab.save_log("T1-05_canvas_draft", canvas)
lab.save_log("T1-05_gold_pairs", {"pairs": gold_pairs})
print("draft and pairs saved — finish overnight in cards/usecase_canvas.md")


## Overnight 1 (20–30 min, no GPU)

Finish the canvas, including the acceptance rule and test set plan. Write six gold pairs (at least two near misses that a careless system would get wrong). Three remaining questions for Day 2. No home GPU. Do not fetch ministry files. If the lab was unfinished, also open `completed/T1-05_canvas_example.md`.

You can re-read T1-01a without starting servers: the point is the chain data → weights → runtime → API → output, and which of those you own. Tomorrow you add **retrieval** (approved handbook chunks) beside those frozen weights — you still will not train.

If you loaded a reference file, also see `completed/T1-05_gold_pairs_example.md`. Optional reading: `cards/t1_05_release_and_rollout.md` (how a stop rule becomes a safe rollout).


## [OPTIONAL] After the canvas: release safety and rollout

For those who want more. Not needed for Day 2. Read `cards/t1_05_release_and_rollout.md`. Short version: a per-document hold protects one CV; a sampled audit estimates how often misses happen across many; a canary limits how many people one bad version can reach. None of them is permission to release, and none can un-send a CV that already leaked.


In [ ]:
# [OPTIONAL] Two calculators. Every input is your own assumption, not a measured price or rate.
for n in (7, 40, 100, 300):
    print(f"0 failures in {n:>3} random audits -> true rate could still be up to {R.upper_bound_95(0, n):.1%} (one-sided 95%)")
print("(Valid only if the audited documents are a random sample of the documents you care about.)\n")

# Scoped cost per acceptable answer for the benchmark run in cell 8: compute + review of held docs only.
print(R.cost_per_acceptable(rows, compute_cost_per_doc=0.0,      # [CHANGE] your figure
                            review_minutes_per_hold=5,           # [CHANGE] your figure
                            reviewer_cost_per_hour=0.0))         # [CHANGE] your figure
